In [ ]:
import pandas as pd
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
import torch
import torch.nn.functional as F
import torch
torch.manual_seed(0)

data_folder = "./Training_Data"
n_inputs = 1000

fits_data = []

for i in tqdm(range(0,n_inputs)):
    fits_file = fits.open(f"{data_folder}/input_image_{i}.fits")
    fits_data.append(np.array(fits_file)[0].data)


fits_data = np.array(fits_data)

train_data = torch.load('./train_data_saved_3000_observations.pt')
test_data = torch.load('./test_data_saved_3000_observations.pt')

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EuclideanLoss(nn.Module):
    def __init__(self, alpha=3, beta=1, gamma=1):
        """
        Custom loss function combining Euclidean loss for RA and Dec
        with L1 loss for Flux predictions.

        Parameters:
        - alpha: Weight for the Euclidean loss (RA and Dec).
        - beta: Weight for the L1 loss of Flux.
        - gamma: Optional additional scaling for Flux loss (if needed).
        """
        super(EuclideanLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, predictions, targets):
        """
        Compute the Euclidean loss for RA and Dec, and L1 loss for Flux.

        Parameters:
        - predictions: Tensor of predicted RA, Dec, and Flux (batch_size x 3).
        - targets: Tensor of true RA, Dec, and Flux (batch_size x 3).

        Returns:
        - combined_loss: Weighted combination of Euclidean and L1 losses.
        """
        predictions, targets = predictions.float(), targets.float()

        # Separate RA, Dec, and Flux predictions and targets
        ra_pred, ra_true = predictions[:, 0], targets[:, 0]
        dec_pred, dec_true = predictions[:, 1], targets[:, 1]
        flux_pred, flux_true = predictions[:, 2], targets[:, 2]

        # Calculate Euclidean distance loss for RA and Dec
        euclidean_distance = torch.sqrt((ra_pred - ra_true) ** 2 + (dec_pred - dec_true) ** 2)
        euclidean_loss = euclidean_distance.mean()

        # Calculate L1 loss for Flux
        flux_loss = F.l1_loss(flux_pred, flux_true)

        # Combine losses with weights
        combined_loss = self.alpha * euclidean_loss + self.beta * flux_loss

        return combined_loss


In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, BatchNorm, AttentionalAggregation, GCNConv

class GraphRegressor(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels=None, num_layers=4, dropout=0.3):
        super(GraphRegressor, self).__init__()

        self.dropout = dropout

        # Convolutional layers
        self.conv_layers = torch.nn.ModuleList()
        self.bn_layers = torch.nn.ModuleList()
        self.residual_layers = torch.nn.ModuleList()

        # Initial convolution layer
        self.conv_layers.append(SAGEConv(in_channels, hidden_channels))
        self.bn_layers.append(BatchNorm(hidden_channels))
        if in_channels != hidden_channels:
            # Linear layer to project input to hidden_channels for residual connection
            self.residual_layers.append(torch.nn.Linear(in_channels, hidden_channels))
        else:
            self.residual_layers.append(torch.nn.Identity())

        # Additional layers
        for _ in range(num_layers - 1):
            self.conv_layers.append(SAGEConv(hidden_channels, hidden_channels))
            self.bn_layers.append(BatchNorm(hidden_channels))
            self.residual_layers.append(torch.nn.Identity())  # Dimensions match; use identi


        self.final_conv = SAGEConv(hidden_channels, 3)  # Outputs RA, Dec, and Flux as a 3-dimensional vector

        # Pooling layer
        self.att_pool = AttentionalAggregation(gate_nn=torch.nn.Linear(3, 1))


    def forward(self, x, edge_index, batch, edge_attr=None):
        # Apply graph convolutional layers with residual connections
        for conv, bn,res in zip(self.conv_layers, self.bn_layers, self.residual_layers):
            if isinstance(res, torch.nn.Identity):
                x_res = x  # Directly pass through input for Identity
            else:
                x_res = res(x)

            if edge_attr is not None:
                x = conv(x, edge_index, edge_attr)
            else:
                x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res  # Add residual connection

        # Global attention pooling to get graph-level representation
        # Final graph convolution to predict RA, Dec, and Flux
        out = self.final_conv(x, edge_index)
        out = F.relu(out)
        # Apply global attention pooling to aggregate node-level predictions
        out = self.att_pool(out, batch)  # Shape: [batch_size, 3]


        return out


In [ ]:
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = 6
hidden_channels = 32
out_channels = 3
model = GraphRegressor(in_channels, hidden_channels,out_channels,num_layers=2).to(device)
loss_fn = EuclideanLoss().to(device)
optimizer = torch.optim.Adam(model.parameters())

num_epochs = 2000
train_losses = {"total": [], "ra": [], "dec": [], "flux": []}

model.train()
for i in range(num_epochs):
    optimizer.zero_grad()

    out = model(train_data.x, train_data.edge_index, train_data.batch)
    total_loss = loss_fn(out, train_data.y)
    total_loss.backward()

    optimizer.step()


    # Track loss values
    train_losses["total"].append(total_loss.item())

    if i % 10 == 0:
        with torch.no_grad():
            test_out = model(test_data.x, test_data.edge_index, test_data.batch)

            ra_loss =  F.l1_loss(test_out[:,::3], test_data.y[:, ::3])
            dec_loss =  F.l1_loss(test_out[:,1::3], test_data.y[:, 1::3])
            flux_loss = F.l1_loss(test_out[:,2::3], test_data.y[:, 2::3])

            train_losses["ra"].append(ra_loss.item())
            train_losses["dec"].append(dec_loss.item())
            train_losses["flux"].append(flux_loss.item())

    if i % 10 == 0:
        print(f"Iteration {i} - Total Loss: {total_loss.item():.4f}, RA Loss: {ra_loss.item():.4f}, Dec Loss: {dec_loss.item():.4f}, Flux Loss: {flux_loss.item():.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_output_comparison(predicted_data, actual_data, cmap='viridis', alpha=0.7):
    """
    Plot predicted and actual astronomical data on a sky map using RA, Dec, and flux.

    Args:
    predicted_data (list): Flat list of predicted data points [RA1, Dec1, Flux1, RA2, Dec2, Flux2, ...]
    actual_data (list): Flat list of actual data points [RA1, Dec1, Flux1, RA2, Dec2, Flux2, ...]
    cmap (str): Colormap for flux representation
    alpha (float): Transparency level for markers
    """

    # Validate input lengths
    if len(predicted_data) % 3 != 0 or len(actual_data) % 3 != 0:
        raise ValueError("Data length must be a multiple of 3 (RA, Dec, Flux per point).")

    # Parse predicted data
    ra_pred = predicted_data[::3]
    dec_pred = predicted_data[1::3]
    flux_pred = predicted_data[2::3]

    # Parse actual data
    ra_act = actual_data[::3]
    dec_act = actual_data[1::3]
    flux_act = actual_data[2::3]

    # Normalize flux for scaling point sizes
    max_flux = max(max(flux_pred), max(flux_act))
    if max_flux > 0:
        size_pred = np.array(flux_pred) / max_flux * 100
        size_act = np.array(flux_act) / max_flux * 100
    else:
        size_pred = np.ones_like(flux_pred) * 10
        size_act = np.ones_like(flux_act) * 10

    plt.figure(figsize=(10, 8))

    # Plot predicted data
    scatter_pred = plt.scatter(ra_pred, dec_pred, s=size_pred, c=flux_pred, cmap=cmap,
                               alpha=alpha, marker='o', edgecolors='black', label='Predicted')

    # Plot actual data
    scatter_act = plt.scatter(ra_act, dec_act, s=size_act, c=flux_act, cmap=cmap,
                              alpha=alpha, marker='^', edgecolors='black', label='Actual')

    # Add colorbar
    cbar = plt.colorbar(scatter_pred)
    cbar.set_label('Flux')

    plt.xlabel('Right Ascension (degrees)')
    plt.ylabel('Declination (degrees)')
    plt.title('Sky Model: Predicted vs Actual')
    #plt.gca().invert_yaxis()  # Optional: Invert y-axis to match astronomical convention
    plt.ylim([0,1])
    plt.xlim([0,1])
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Example usage:
plot_output_comparison(test_out[1].cpu().detach().numpy(), test_data.y[1].cpu().detach().numpy())